# AutoQuant

This notebook shows a working code example of how to use AIMET AutoQuant feature.

AIMET offers a suite of neural network post-training quantization (PTQ) techniques that can be applied in succession. However, the process of finding the right combination and sequence of techniques to apply is time-consuming and requires careful analysis, which can be challenging especially for non-expert users. We instead recommend AutoQuant to save time and effort.

AutoQuant is an API that applies various PTQ techniques in AIMET automatically based on analyzing the model and best-known heuristics. In AutoQuant, users specify the amount of tolerable accuracy drop, and AutoQuant will apply PTQ techniques cumulatively until the target accuracy is satisfied.


#### Overall flow
This notebook covers the following
1. Define constants and helper functions
2. Load a pretrained FP32 model
3. Run AutoQuant

#### What this notebook is not
This notebook is not designed to show state-of-the-art AutoQuant results. For example, it uses a relatively quantization-friendly model like Resnet18. Also, some optimization parameters are deliberately chosen to have the notebook execute more quickly.


---
## Dataset

This notebook relies on the ImageNet dataset for the task of image classification. If you already have a version of the dataset readily available, please use that. Else, please download the dataset from appropriate location (e.g. https://image-net.org/challenges/LSVRC/2012/index.php#).

**Note1**: The ImageNet dataset typically has the following characteristics and the dataloader provided in this example notebook rely on these
- Subfolders 'train' for the training samples and 'val' for the validation samples. Please see the [pytorch dataset description](https://pytorch.org/vision/0.8/_modules/torchvision/datasets/imagenet.html) for more details.
- A subdirectory per class, and a file per each image sample

**Note2**: To speed up the execution of this notebook, you may use a reduced subset of the ImageNet dataset. E.g. the entire ILSVRC2012 dataset has 1000 classes, 1000 training samples per class and 50 validation samples per class. But for the purpose of running this notebook, you could perhaps reduce the dataset to say 2 samples per class. This exercise is left upto the reader and is not necessary.

Edit the cell below and specify the directory where the downloaded ImageNet dataset is saved.

In [2]:
import cv2
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import json
from mmcv.transforms import Compose
import numpy as np
from mmdet.utils import get_test_pipeline_cfg

def read_json(json_path):
    with open(json_path) as f:
        data = json.load(f)
    return data

def preprocess(test_pipeline, image):
    if isinstance(image, np.ndarray):
        # Calling this method across libraries will result
        # in module unregistered error if not prefixed with mmdet.
        test_pipeline[0].type = 'mmdet.LoadImageFromNDArray'
    test_pipeline = Compose(test_pipeline)
    return test_pipeline(dict(img=image))


## 1. Define Constants and Helper functions

In this section the constants and helper functions needed to run this eaxmple are defined.

- **EVAL_DATASET_SIZE** A typical value is 5000. In this notebook, this value has been set to 500 for faster execution.
- **CALIBRATION_DATASET_SIZE** A typical value is 2000. In this notebook, this value has been set to 200 for faster execution.


The helper function **_create_sampled_data_loader()** returns a DataLoader based on the dataset and the number of samples provided.

## 2. Load a pretrained FP32 model
For this example, we are going to load a pretrained resnet18 model from torchvision. Similarly, you can load any pretrained PyTorch model instead.

In [3]:
# %cd /content/drive/MyDrive/Aimet-torch/mmdetection-3.3.0/
import torch
from mmdet.apis import DetInferencer

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize([640, 640]),  # Resize
])

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
CONFIG_PATH = '/teamspace/studios/this_studio/mmdetection/rtmdet_tiny_8xb32-300e_coco.py'
WEIGHTS_PATH = '/teamspace/studios/this_studio/mmdetection/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth'

ROOT_DATASET_DIR = '/teamspace/studios/aimet/COCO'
IMAGES_DIR = os.path.join(ROOT_DATASET_DIR, 'images')
ANNOTATIONS_JSON_PATH = os.path.join(ROOT_DATASET_DIR, 'annotations/instances_val2017.json')
# ANNOTATIONS_JSON_PATH = "/home/shayaan/Desktop/aimet/my_mmdet/temp.json"

model = DetInferencer(model=CONFIG_PATH, weights=WEIGHTS_PATH, device=DEVICE)

DEVICE

[2024-08-19 13:27:04,578] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH


/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


 [WARNING]  NVIDIA Inference is only supported on Ampere and newer architectures
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.2
 [WARNING]  using untested triton version (2.2.0), only 1.0.0 is known to be compatible
2024-08-19 13:27:05,871 - root - INFO - AIMET
Loads checkpoint by local backend from path: /teamspace/studios/this_studio/mmdetection/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth
The model and loaded state dict do not match exactly

unexpected key in source state_dict: data_preprocessor.mean, data_preprocessor.std

08/19 13:27:12 - mmengine - WARNING - Failed to search registry with scope "mmdet" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmdet" is a correct scope, or whether the registry is initialized.


/teamspace/studios/this_studio/mmengine/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.LocalVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backend.__class__}, '


device(type='cuda', index=0)

In [4]:
from mmcv.transforms import Compose
test_evaluator = model.cfg.test_evaluator
test_evaluator.type = 'mmdet.evaluation.CocoMetric' 
test_evaluator.dataset_meta = model.model.dataset_meta
test_evaluator.ann_file = ANNOTATIONS_JSON_PATH
test_evaluator = Compose(test_evaluator)

loading annotations into memory...


Done (t=0.80s)
creating index...
index created!


In [5]:
from mmdet.models.utils import samplelist_boxtype2tensor
from mmengine.registry import MODELS

collate_preprocessor = model.preprocess
predict_by_feat = model.model.bbox_head.predict_by_feat
rescale = True

preprocessor = MODELS.build(model.cfg.model.data_preprocessor)
def add_pred_to_datasample(data_samples, results_list):
    for data_sample, pred_instances in zip(data_samples, results_list):
        data_sample.pred_instances = pred_instances
    samplelist_boxtype2tensor(data_samples)
    return data_samples

In [6]:
import random
from typing import Optional
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from mmengine.structures import InstanceData

EVAL_DATASET_SIZE = 5000
CALIBRATION_DATASET_SIZE = 2000
BATCH_SIZE = 40

class CustomImageDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, annotations_json_path, transform=None):
        self.transform = transform
        self.images_dir = images_dir
        self.annotations_json = read_json(annotations_json_path)


    def __len__(self):
        return len(self.annotations_json['images'])

    def __getitem__(self, idx):
        image_dict = self.annotations_json['images'][idx]
        image_path = os.path.join(self.images_dir, image_dict['file_name'])
        image_id = image_dict['id']


        pre_processed = collate_preprocessor(inputs=[image_path], batch_size=BATCH_SIZE)
        _, data = list(pre_processed)[0]
        data = preprocessor(data, False)['inputs'][0]

        return image_id, image_path, data

evalDataset = CustomImageDataset(images_dir=IMAGES_DIR, annotations_json_path=ANNOTATIONS_JSON_PATH, transform=transform)

In [7]:
def eval_callback(model: torch.nn.Module, num_samples: Optional[int] = None) -> float:    
    data_loader = DataLoader(evalDataset, batch_size=2)

    new_preds = []
    for image_id, image_path, _ in tqdm(data_loader):
        pre_processed = collate_preprocessor(inputs=image_path, batch_size=BATCH_SIZE)
        _, data = list(pre_processed)[0]
        data = preprocessor(data, False)

        preds = model.model._forward(data['inputs'].cuda())
        for p in preds:
            for t in p:
                print(t.shape)
            print("(((((())))))")
        batch_img_metas = [
        data_samples.metainfo for data_samples in data['data_samples']
        ]
        preds = predict_by_feat(*preds, batch_img_metas=batch_img_metas, rescale=True)
        print(type(preds))
        print(preds.shape)
        preds = add_pred_to_datasample(data['data_samples'], preds)
        
        for img_id, pred in zip(image_id, preds):
            pred = pred.pred_instances
            new_pred = InstanceData(metainfo={"img_id": int(img_id)})
            new_pred.bboxes = [np.array(p) for p in pred['bboxes'].cpu()]
            new_pred.labels = pred['labels'].cpu()
            new_pred.scores = pred['scores'].cpu()
            new_preds.append(new_pred)
    
    eval_results = test_evaluator(new_preds)
    bbox_map = eval_results['bbox_mAP']
    # with open("/teamspace/studios/this_studio/mmdetection/ptq_stats/eval_acc_fp32.json", "w") as f:
    #     json.dump(eval_results, f, indent=4)
    return bbox_map

In [8]:
from aimet_torch.quantsim import QuantizationSimModel, QuantScheme
from aimet_torch.model_preparer import prepare_model

if DEVICE.type == "cpu":
    dummy_input = torch.rand(1, 3, 640, 640)
else:
    dummy_input = torch.rand(1, 3, 640, 640).cuda()



prepared_model = prepare_model(model.model)

sim = QuantizationSimModel(model=prepared_model,
                        quant_scheme=QuantScheme.post_training_tf_enhanced,
                        dummy_input=dummy_input,
                        default_output_bw=8,
                        default_param_bw=8,)


STAGE DET FUNC CALLED
2024-08-19 13:27:14,573 - ModelPreparer - INFO - Functional         : Adding new module for node: {backbone.stem.0.bn.module_batch_norm} 
2024-08-19 13:27:14,574 - ModelPreparer - INFO - Functional         : Adding new module for node: {backbone.stem.1.bn.module_batch_norm_1} 
2024-08-19 13:27:14,576 - ModelPreparer - INFO - Functional         : Adding new module for node: {backbone.stem.2.bn.module_batch_norm_2} 
2024-08-19 13:27:14,577 - ModelPreparer - INFO - Functional         : Adding new module for node: {backbone.stage1.0.bn.module_batch_norm_3} 
2024-08-19 13:27:14,578 - ModelPreparer - INFO - Functional         : Adding new module for node: {backbone.stage1.1.short_conv.bn.module_batch_norm_4} 
2024-08-19 13:27:14,579 - ModelPreparer - INFO - Functional         : Adding new module for node: {backbone.stage1.1.main_conv.bn.module_batch_norm_5} 
2024-08-19 13:27:14,580 - ModelPreparer - INFO - Functional         : Adding new module for node: {backbone.stage

2024-08-19 13:27:14,606 - ModelPreparer - INFO - Functional         : Adding new module for node: {backbone.stage4.2.blocks.0.conv2.depthwise_conv.bn.module_batch_norm_30} 
2024-08-19 13:27:14,977 - ModelPreparer - INFO - Functional         : Adding new module for node: {backbone.stage4.2.blocks.0.conv2.pointwise_conv.bn.module_batch_norm_31} 
2024-08-19 13:27:14,979 - ModelPreparer - INFO - Functional         : Adding new module for node: {backbone.stage4.2.module_cat_4} 
2024-08-19 13:27:14,979 - ModelPreparer - INFO - Functional         : Adding new module for node: {backbone.stage4.2.attention.module_mul_3} 
2024-08-19 13:27:14,980 - ModelPreparer - INFO - Functional         : Adding new module for node: {backbone.stage4.2.final_conv.bn.module_batch_norm_32} 
2024-08-19 13:27:14,981 - ModelPreparer - INFO - Functional         : Adding new module for node: {neck.reduce_layers.0.bn.module_batch_norm_33} 
2024-08-19 13:27:14,982 - ModelPreparer - INFO - Functional         : Adding new

In [9]:
# print(str(sim))

In [10]:
print("CALCULATING FP32 ACCURACY NOW")
accuracy = eval_callback(model)

CALCULATING FP32 ACCURACY NOW


  0%|          | 0/2500 [00:00<?, ?it/s]

torch.Size([2, 80, 80, 80])
torch.Size([2, 80, 40, 40])
torch.Size([2, 80, 20, 20])
(((((())))))
torch.Size([2, 4, 80, 80])
torch.Size([2, 4, 40, 40])
torch.Size([2, 4, 20, 20])
(((((())))))


/usr/local/lib/python3.10/dist-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
  0%|          | 0/2500 [00:00<?, ?it/s]

<class 'list'>


AttributeError: 'list' object has no attribute 'shape'

In [10]:
# print(f'- FP32 accuracy: {accuracy}')

## 3. Run AutoQuant
### Create AutoQuant Object

The AutoQuant feature utilizes an unlabeled dataset to achieve quantization. The class **UnlabeledDatasetWrapper** creates an unlabeled Dataset object from a labeled Dataset.

In [11]:
class UnlabeledDatasetWrapper(Dataset):
    def __init__(self, dataset, num_samples):
        self._dataset = dataset
        self.num_samples = num_samples if num_samples < len(self._dataset) else len(self._dataset)
        self.transform = transform

    def __len__(self):
        return self.num_samples 

    def __getitem__(self, index):
        _, _, transformed_images = self._dataset[index]
        return transformed_images
    

In [12]:
from aimet_torch.auto_quant import AutoQuant
from glob import glob

def new_eval_callback(model: torch.nn.Module, num_samples: Optional[int] = None) -> float:
    data_loader = DataLoader(evalDataset, batch_size=BATCH_SIZE)
    new_preds = []
    for image_id, image_path, _ in tqdm(data_loader):
        pre_processed = collate_preprocessor(inputs=image_path, batch_size=BATCH_SIZE)
        _, data = list(pre_processed)[0]
        data = preprocessor(data, False)
        preds = model(data['inputs'].cuda())
        print(preds[0].shape)
        print(preds[1].shape)
        print(preds[2].shape)
        batch_img_metas = [
        data_samples.metainfo for data_samples in data['data_samples']
        ]
        preds = predict_by_feat(*preds, batch_img_metas=batch_img_metas, rescale=True)
        preds = add_pred_to_datasample(data['data_samples'], preds)
        
        for img_id, pred in zip(image_id, preds):
            pred = pred.pred_instances
            new_pred = InstanceData(metainfo={"img_id": int(img_id)})
            new_pred.bboxes = [np.array(p) for p in pred['bboxes'].cpu()]
            new_pred.labels = pred['labels'].cpu()
            new_pred.scores = pred['scores'].cpu()
            new_preds.append(new_pred)

    eval_results = test_evaluator(new_preds)
    num_file = len(glob("/teamspace/studios/this_studio/aimet/Examples/torch/quantization/disbaled_eval_stats/eval_acc_quant_*"))
    with open(f"/teamspace/studios/this_studio/aimet/Examples/torch/quantization/disbaled_eval_stats/eval_acc_quant_{num_file}.json", "w") as f:
        json.dump(eval_results, f, indent=4)
    bbox_map = eval_results['bbox_mAP']
    return bbox_map


calibration_data_loader = DataLoader(UnlabeledDatasetWrapper(evalDataset, num_samples=1))
auto_quant = AutoQuant(prepared_model,
                       dummy_input=dummy_input,
                       data_loader=calibration_data_loader,
                       eval_callback=new_eval_callback)

### Run AutoQuant Inference
This step runs AutoQuant inference. AutoQuant inference will run evaluation using the **eval_callback** with the vanilla quantized model without applying PTQ techniques. This will be useful for measuring the baseline evaluation score before running AutoQuant optimization.

In [13]:
modules_to_ignore = ['module_batch_norm_14', 'module_batch_norm_7', 'backbone.stage2.1.blocks.0.conv2.pointwise_conv.conv', 'module_batch_norm_21', 'module_batch_norm_30', 'module_batch_norm_37', 'module_batch_norm_44', 'module_batch_norm_51', 'module_batch_norm_58']


In [14]:
sim, initial_accuracy = auto_quant.run_inference()
# model = auto_quant.run_inference()


2024-08-19 12:45:19,167 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_batch_norm} 


2024-08-19 12:45:19,169 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_mul} 


2024-08-19 12:45:19,172 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_batch_norm_1} 


2024-08-19 12:45:19,175 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_mul_1} 


2024-08-19 12:45:19,177 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_batch_norm_2} 


...

  0%|          | 0/125 [00:01<?, ?it/s]

torch.Size([40, 96, 80, 80])
torch.Size([40, 96, 40, 40])
torch.Size([40, 96, 20, 20])



| Batchnorm Folding


Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/aimet_torch/auto_quant.py", line 334, in run_inference
    sess.set_ptq_result(model=model,
  File "/usr/local/lib/python3.10/dist-packages/aimet_torch/auto_quant.py", line 1145, in set_ptq_result
    acc = self._eval_func(sim.model)
  File "/usr/local/lib/python3.10/dist-packages/aimet_torch/auto_quant.py", line 317, in _evaluate_model_performance
    return self.eval_callback(model, NUM_SAMPLES_FOR_PERFORMANCE_EVALUATION)
  File "/usr/local/lib/python3.10/dist-packages/aimet_torch/auto_quant.py", line 271, in eval_callback_wrapper
    return eval_callback(model, *args, **kwargs)
  File "/tmp/ipykernel_148707/2206052341.py", line 18, in new_eval_callback
    preds = predict_by_feat(*preds, batch_img_metas=batch_img_metas, rescale=True)
  File "/teamspace/studios/this_studio/mmdetection/mmdet/models/dense_heads/base_dense_head.py", line 261, in predict_by_feat
    mlvl_priors = self.prior_generator.grid_

AssertionError: 

In [15]:
# print(f"- Quantized A ccuracy (before optimization): {initial_accuracy}")

- Quantized A ccuracy (before optimization): 0.386


### Set AdaRound Parameters (optional)
AutoQuant uses a set of predefined default parameters for AdaRound.
These values were determined empirically and work well with the common models.
However, if necessary, you can also use your custom parameters for Adaround.
In this notebook, we will use very small AdaRound parameters for faster execution.

In [12]:
from aimet_torch.adaround.adaround_weight import AdaroundParameters

adaround_params = AdaroundParameters(calibration_data_loader, num_batches=len(calibration_data_loader), default_num_iterations=2000)
auto_quant.set_adaround_params(adaround_params)

### Run AutoQuant Optimization
This step runs AutoQuant optimization, which returns the best possible quantized model, corresponding evaluation score and the path to the encoding file.
The **allowed_accuracy_drop** parameter indicates the tolerable amount of accuracy drop. AutoQuant applies a series of quantization features until the target accuracy (FP32 accuracy - allowed accuracy drop) is satisfied. When the target accuracy is reached, AutoQuant will return immediately without applying furhter PTQ techniques. Please refer AutoQuant User Guide and API documentation for complete details.

In [13]:
model, optimized_accuracy, encoding_path = auto_quant.optimize(allowed_accuracy_drop=0.2)
print(f"- Quantized Accuracy (after optimization):  {optimized_accuracy}")

2024-08-13 08:30:08,469 - AutoQuant - INFO - Starting AutoQuant


  0%|          | 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
100%|██████████| 125/125 [06:47<00:00,  3.26s/it]


08/13 08:37:30 - mmengine - INFO - Evaluating bbox...
Loading and preparing results...
DONE (t=5.61s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=79.97s).
Accumulating evaluation results...
DONE (t=25.30s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.411
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.579
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.447
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.210
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.455
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.583
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.334
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.554
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.605
 Average Rec

2024-08-13 08:39:33,597 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_batch_norm} 


2024-08-13 08:39:33,601 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_mul} 


...

2024-08-13 09:19:26,389 - Quant - INFO - Unsupported op type Mean


2024-08-13 09:19:26,405 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:default


 97%|█████████▋| 121/125 [16:24<00:32,  8.14s/it]
| QuantScheme Selection


Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/aimet_torch/auto_quant.py", line 739, in _optimize_main
    self._quantsim_params["quant_scheme"] = sess.wrap(self._choose_default_quant_scheme)()
  File "/usr/local/lib/python3.10/dist-packages/aimet_torch/auto_quant.py", line 1073, in wrapper
    ret = fn(*args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/aimet_torch/auto_quant.py", line 721, in _choose_default_quant_scheme
    return max(candidates, key=eval_fn)
  File "/usr/local/lib/python3.10/dist-packages/aimet_torch/auto_quant.py", line 688, in eval_fn
    eval_score = self._evaluate_model_performance(sim.model)
  File "/usr/local/lib/python3.10/dist-packages/aimet_torch/auto_quant.py", line 335, in _evaluate_model_performance
    return self.eval_callback(model, NUM_SAMPLES_FOR_PERFORMANCE_EVALUATION)
  File "/usr/local/lib/python3.10/dist-packages/aimet_torch/auto_quant.py", line 289, in eval_callback_wrapper
    return eval_callb

KeyboardInterrupt: 

In [ ]:
# model, optimized_accuracy, encoding_path


---
## Summary

Hope this notebook was useful for you to understand how to use AIMET AutoQuant feature.

Few additional resources
- Refer to the AIMET API docs to know more details of the APIs and parameters
- Refer to the other example notebooks to understand how to use AIMET CLE and AdaRound features in a standalone fashion.